In [1]:
import numpy as np
from plotly.io import show

from skfolio.datasets import load_sp500_dataset
from skfolio.optimization import MeanRisk
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()
prices = prices[["AAPL", "GE", "JPM"]]

X = prices_to_returns(prices)

In [2]:
model = MeanRisk()
model.fit(X)
print(sum(model.weights_))
model.weights_

0.9999999999999999


array([0.22768876, 0.56566507, 0.20664617])

In [3]:
model = MeanRisk(budget=0.5)
model.fit(X)
print(sum(model.weights_))
model.weights_

0.5


array([0.11391513, 0.28246101, 0.10362386])

In [4]:
model = MeanRisk(budget=None, min_budget=0.3, max_budget=0.5)
model.fit(X)
print(sum(model.weights_))
model.weights_

0.30000034617916516


array([0.06832987, 0.16956647, 0.06210401])

In [5]:
#lower and upper bounds on weights
model = MeanRisk(budget=-1, min_weights=-1)
model.fit(X)
print(sum(model.weights_))
model.weights_

-1.0000000000000002


array([-0.22770271, -0.56559255, -0.20670474])

In [6]:
model = MeanRisk(min_weights=[0, 0.5, 0.1])
model.fit(X)
print(sum(model.weights_))
model.weights_

1.0


array([0.22788246, 0.56548525, 0.20663228])

In [7]:
portfolio = model.predict(X)
fig = portfolio.plot_composition()
show(fig)

In [8]:
model = MeanRisk(min_weights={"GE": 0.5, "JPM": 0.1})
model.fit(X)
print(sum(model.weights_))
model.weights_

1.0


array([0.22788246, 0.56548525, 0.20663228])

In [9]:
model = MeanRisk(budget=3, max_weights=1.5)
model.fit(X)
print(sum(model.weights_))
model.weights_

3.0000000000000018


array([0.74197781, 1.49999867, 0.75802352])

In [10]:
model = MeanRisk(min_weights=-1, max_short=0.5)
model.fit(X)
print(sum(model.weights_))
model.weights_

0.9999999999999999


array([0.22770146, 0.56558315, 0.20671539])

In [11]:
#group and liear constraints 
groups = {
    "AAPL": ["Technology", "Mega Cap"],
    "GE": ["Industrial", "Big Cap"],
    "JPM": ["Financial", "Big Cap"],
}
# You can also provide a 2D array-like:
# groups = [["Technology", "Industrial", "Financial"], ["Mega Cap", "Big Cap", "Big Cap"]]
linear_constraints = [
    "Technology + 1.5 * Industrial <= 2 * Financial",  # First group
    "Mega Cap >= 0.75 * Big Cap",  # Second group
    "Technology >= Big Cap",  # Mix of first and second groups
    "Mega Cap >= 2 * JPM",  # Mix of groups and assets
]
# Note that only the first constraint would be sufficient in that case.

model = MeanRisk(groups=groups, linear_constraints=linear_constraints)
model.fit(X)
model.weights_


array([6.66666667e-01, 1.17341989e-11, 3.33333333e-01])

In [12]:
#left and right inequalities 
left_inequality = np.array(
    [[1.0, 1.5, -2.0], [-1.0, 0.75, 0.75], [-1.0, 1.0, 1.0], [-1.0, -0.0, 2.0]]
)
right_inequality = np.array([0.0, 0.0, 0.0, 0.0])

model = MeanRisk(left_inequality=left_inequality, right_inequality=right_inequality)
model.fit(X)
model.weights_

array([6.66666667e-01, 1.17341989e-11, 3.33333333e-01])